# 02 · Mínimos cuadrados y correlación de Pearson (secciones 5.8 y 5.9)

X = `ANTIGUEDAD_MESES`, Y = `VAL_CFM`. Se usan los pares individuales válidos (no marcas de clase).

- Pendiente: b = [nΣxy − ΣxΣy] / [nΣx² − (Σx)²]  ·  Intercepto: a = ȳ − b·x̄
- Correlación: r = [nΣxy − ΣxΣy] / √([nΣx² − (Σx)²]·[nΣy² − (Σy)²])  ·  R² = r²

Se calcula a mano con las sumas y se valida con `scipy.stats.linregress`. El ajuste describe asociación; no
demuestra causalidad.

In [1]:
import math
import numpy as np
import pandas as pd
from scipy import stats
import config

el = config.leer("elegibles")
pares = el[["ANTIGUEDAD_MESES", "VAL_CFM"]].dropna().astype(float)
xv, yv = pares.ANTIGUEDAD_MESES.to_numpy(), pares.VAL_CFM.to_numpy()
n = len(pares)
print(f"Pares válidos: n = {n:,} (se excluyen {len(el) - n} registros sin antigüedad)")

Pares válidos: n = 5,227 (se excluyen 3 registros sin antigüedad)


## Sumas y cálculo a mano

In [2]:
Sx, Sy = float(xv.sum()), float(yv.sum())
Sxy, Sxx, Syy = float((xv * yv).sum()), float((xv ** 2).sum()), float((yv ** 2).sum())
num = n * Sxy - Sx * Sy
den_x = n * Sxx - Sx ** 2
den_y = n * Syy - Sy ** 2
b = num / den_x
xbar, ybar = Sx / n, Sy / n
a = ybar - b * xbar
r = num / math.sqrt(den_x * den_y)
R2 = r ** 2
sumas = {"n": n, "Sx": Sx, "Sy": Sy, "Sxy": Sxy, "Sxx": Sxx, "Syy": Syy, "num": num, "den_x": den_x, "den_y": den_y,
         "xbar": xbar, "ybar": ybar}
print(f"b = {b:.4f} pesos por mes · a = {a:,.2f} · r = {r:.4f} · R² = {R2:.4f}")
pd.Series(sumas)

b = -50.6053 pesos por mes · a = 39,725.85 · r = -0.2488 · R² = 0.0619


n        5.227000e+03
Sx       4.767500e+04
Sy       2.052344e+08
Sxy      1.610149e+09
Sxx      5.607741e+06
Syy      8.272401e+12
num     -1.368304e+12
den_x    2.703876e+10
den_y    1.118674e+15
xbar     9.120911e+00
ybar     3.926429e+04
dtype: float64

## Validación con scipy

In [3]:
lr = stats.linregress(xv, yv)
validacion = [
    {"indicador": "Pendiente b", "manual": b, "computacional": lr.slope, "tolerancia": 1e-6},
    {"indicador": "Intercepto a", "manual": a, "computacional": lr.intercept, "tolerancia": 1e-4},
    {"indicador": "Correlación r", "manual": r, "computacional": lr.rvalue, "tolerancia": 1e-9},
    {"indicador": "R²", "manual": R2, "computacional": lr.rvalue ** 2, "tolerancia": 1e-9},
    {"indicador": "r (numpy.corrcoef)", "manual": r, "computacional": float(np.corrcoef(xv, yv)[0, 1]), "tolerancia": 1e-9},
]
for v in validacion:
    v["diferencia"] = abs(v["manual"] - v["computacional"])
    v["cumple"] = bool(v["diferencia"] <= v["tolerancia"])
    v["observacion"] = "sumas manuales vs scipy.stats.linregress" if "numpy" not in v["indicador"] else "vs numpy"
pd.DataFrame(validacion)

,indicador,manual,computacional,tolerancia,diferencia,cumple,observacion
0,Pendiente b,-50.605289,-50.605289,1.000000e-06,4.263256e-14,True,sumas manuales vs scipy.stats.linregress
1,Intercepto a,39725.851382,39725.851382,1.000000e-04,0.000000e+00,True,sumas manuales vs scipy.stats.linregress
2,Correlación r,-0.248793,-0.248793,1.000000e-09,1.942890e-16,True,sumas manuales vs scipy.stats.linregress
3,R²,0.061898,0.061898,1.000000e-09,9.714451e-17,True,sumas manuales vs scipy.stats.linregress
4,r (numpy.corrcoef),-0.248793,-0.248793,1.000000e-09,8.049117e-16,True,vs numpy


## Residuos y valores atípicos
Residuo eᵢ = yᵢ − ŷᵢ. Se marcan como atípicos los residuos estandarizados con |z| > 3.

In [4]:
yhat = a + b * xv
res = yv - yhat
z = (res - res.mean()) / res.std(ddof=1)
atipicos = int((np.abs(z) > 3).sum())
print(f"Residuos: media {res.mean():.2e}, desviación {res.std(ddof=1):,.0f}; atípicos |z|>3: {atipicos}")

# ¿Qué pasa sin las líneas con cargo 0 y sin las líneas de 0–3 meses? (sensibilidad, no reemplaza el ajuste principal)
def ajuste(mask, etiqueta):
    s = stats.linregress(xv[mask], yv[mask])
    return {"escenario": etiqueta, "n": int(mask.sum()), "b": s.slope, "a": s.intercept, "r": s.rvalue, "R2": s.rvalue ** 2}


sensibilidad = pd.DataFrame([
    ajuste(np.ones(n, bool), "Todos los pares válidos"),
    ajuste(yv > 0, "Sin cargo fijo 0"),
    ajuste(xv > 3, "Antigüedad mayor a 3 meses"),
    ajuste(np.abs(z) <= 3, "Sin atípicos |z|>3"),
])
sensibilidad.round(4)

Residuos: media 3.36e-12, desviación 6,198; atípicos |z|>3: 167


,escenario,n,b,a,r,R2
0,Todos los pares válidos,5227,-50.6053,39725.8514,-0.2488,0.0619
1,Sin cargo fijo 0,5182,-11.5461,39698.6666,-0.0635,0.0040
2,Antigüedad mayor a 3 meses,1206,-53.3429,40085.1695,-0.2703,0.0730
3,Sin atípicos |z|>3,5060,-26.4229,39898.9435,-0.1705,0.0291


In [5]:
# medias por tramo de antigüedad (ayuda a leer la recta)
tramos = pd.cut(xv, [-1, 3, 12, 36, 120, 10_000], labels=["0–3", "4–12", "13–36", "37–120", ">120"])
por_tramo = (pd.DataFrame({"tramo": tramos, "cfm": yv}).groupby("tramo", observed=False).cfm
               .agg(["size", "mean", "median"]).reset_index())
por_tramo

,tramo,size,mean,median
0,0–3,4021,39586.227804,40185.0
1,4–12,771,40920.670558,42701.0
2,13–36,144,33777.138889,36577.0
3,37–120,141,39088.304965,36829.0
4,>120,150,27553.333333,31418.0


In [6]:
# nube para el dashboard: pares agregados (antigüedad, cargo) con su conteo; sin identificadores
nube = pares.value_counts().rename("conteo").reset_index()
config.guardar_json({
    "n": n, "excluidos": len(el) - n, "sumas": sumas, "b": b, "a": a, "r": r, "R2": R2,
    "scipy": {"slope": lr.slope, "intercept": lr.intercept, "rvalue": lr.rvalue, "pvalue": lr.pvalue, "stderr": lr.stderr},
    "residuos": {"media": float(res.mean()), "sd": float(res.std(ddof=1)), "atipicos": atipicos,
                 "hist": np.histogram(res, bins=40)[0].tolist(), "bordes": np.histogram(res, bins=40)[1].tolist()},
    "sensibilidad": sensibilidad.to_dict("records"), "por_tramo": por_tramo.to_dict("records"),
    "nube": nube.to_numpy().tolist(), "validacion": validacion,
    "x_rango": [float(xv.min()), float(xv.max())],
}, "r02_regresion")
print(f"Puntos distintos en la nube: {len(nube):,}")

Puntos distintos en la nube: 507
